<a href="https://colab.research.google.com/github/Chimera418/dl-endsem-logical-ad/blob/main/notebooks/03_colab_phaseD_fusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase D — Multi-Scale Feature Fusion (`E16_FPN`, `E17_BiFPN`)
**Project:** 22AI304 / 22AIE304 End-Semester Capstone — Team 11  
**Protocol:** Attention-free `PDN-S` + 4-level BN-free multi-scale pyramid (`P2..P5 → 59×59`) under the exact 200-epoch `AdamW` cosine schedule (`patience=10`, native BF16 on A100/L4/G4, FP16 on T4).

- **`E16_FPN`**: Standard unidirectional top-down Feature Pyramid Network (`P5 → P4 → P3 → P2`)
- **`E17_BiFPN`**: Weighted bi-directional Feature Pyramid Network (`P5 → P2` + `P2 → P5` with fast normalized fusion weights & depthwise-separable convs)

**Reference Baselines:** `E01_PUAD_200ep` (`89.64%` Mean / `84.07%` Pooled) | Phase C Winner `E15_CrissCross` (`90.74%` Mean / `85.33%` Pooled).

In [1]:
#@title 1. Verify Colab GPU, RAM, Disk & Install Dependencies
!pip install -q timm thop pyyaml scikit-learn pandas
import torch, psutil, shutil, sys
print('=== COLAB SESSION AUDIT (PHASE D) ===')
print(f'Python        : {sys.version.split()[0]}')
print(f'PyTorch       : {torch.__version__} | CUDA: {torch.version.cuda}')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    native_bf16 = p.major >= 8
    print(f'GPU Model     : {p.name} (CC {p.major}.{p.minor}) | VRAM: {p.total_memory/1024**3:.2f} GiB | Native BF16: {native_bf16}')
else:
    raise RuntimeError('No GPU attached! Runtime -> Change runtime type -> L4 / G4 / A100 GPU.')
ram = psutil.virtual_memory()
disk = shutil.disk_usage('/content')
print(f'System RAM    : {ram.available/1e9:.1f} GB free / {ram.total/1e9:.1f} GB total')
print(f'Local SSD     : {disk.free/1e9:.1f} GB free / {disk.total/1e9:.1f} GB total')

=== COLAB SESSION AUDIT (PHASE D) ===
Python        : 3.13.16
PyTorch       : 2.14.1+cu130 | CUDA: 13.0
GPU Model     : NVIDIA L4 (CC 8.9) | VRAM: 22.03 GiB | Native BF16: True
System RAM    : 54.6 GB free / 56.9 GB total
Local SSD     : 253.0 GB free / 253.1 GB total


In [2]:
#@title 2. Mount Drive, Pull Latest Repo, Unpack Data & Run Section 3 Audit
from google.colab import drive
drive.mount('/content/drive')

import os
REPO_URL = 'https://github.com/Chimera418/dl-endsem-logical-ad.git'
DRIVE_ROOT = '/content/drive/MyDrive/DL_Endsem_Team11'
DRIVE_RUNS = f'{DRIVE_ROOT}/runs'
os.makedirs(DRIVE_RUNS, exist_ok=True)

if not os.path.exists('/content/Endsem/.git'):
    !git clone {REPO_URL} /content/Endsem
else:
    !git -C /content/Endsem pull --ff-only
%cd /content/Endsem

if not os.path.exists('/content/data/mvtec_loco_anomaly_detection/breakfast_box'):
    os.makedirs('/content/data', exist_ok=True)
    !tar -xf "{DRIVE_ROOT}/data/mvtec_loco_anomaly_detection.tar.xz" -C /content/data/
if not os.path.exists('/content/data/imagenet_penalty'):
    os.makedirs('/content/data/imagenet_penalty', exist_ok=True)
    !unzip -q "{DRIVE_ROOT}/data/imagenet_penalty_10k.zip" -d /content/data/imagenet_penalty

os.environ['ENDSEM_LOCO_ROOT'] = '/content/data/mvtec_loco_anomaly_detection'
os.environ['ENDSEM_IMAGENET_ROOT'] = '/content/data/imagenet_penalty'
os.environ['ENDSEM_TEACHER_PATH'] = f'{DRIVE_ROOT}/weights/teacher_pdn_s.pt'

!python scripts/verify_env.py

Mounted at /content/drive
Cloning into '/content/Endsem'...
remote: Enumerating objects: 164, done.
remote: Counting objects: 100% (164/164), done.
remote: Compressing objects: 100% (108/108), done.
remote: Total 164 (delta 89), reused 120 (delta 52), pack-reused 0 (from 0)
Receiving objects: 100% (164/164), 169.14 KiB | 3.60 MiB/s, done.
Resolving deltas: 100% (89/89), done.
/content/Endsem
SECTION 3 — ENVIRONMENT & DATASET VERIFICATION AUDIT
1. CUDA Available    : True
2. GPU Model         : NVIDIA L4 (CC 8.9, BF16=True)
3. GPU Memory        : 22.03 GiB (22563 MiB)
4. PyTorch / CUDA    : PyTorch 2.14.1+cu130 | CUDA 13.0 | cuDNN 92800
   Python / Packages : Python 3.13.16 | timm=1.0.31 | thop=0.1.1 | mamba_ssm=None
5. System RAM        : 50.08 GB free / 52.96 GB total (5.4% used)
6. Disk Space        : 223.53 GB free / 235.68 GB total (/content/Endsem)
------------------------------------------------------------------------------
7. Dataset & Weights Accessibility:
   MVTec LOCO AD   

In [5]:
import os
os.environ['ENDSEM_LOCO_ROOT'] = '/content/data'
!python scripts/verify_env.py

SECTION 3 — ENVIRONMENT & DATASET VERIFICATION AUDIT
1. CUDA Available    : True
2. GPU Model         : NVIDIA L4 (CC 8.9, BF16=True)
3. GPU Memory        : 22.03 GiB (22563 MiB)
4. PyTorch / CUDA    : PyTorch 2.14.1+cu130 | CUDA 13.0 | cuDNN 92800
   Python / Packages : Python 3.13.16 | timm=1.0.31 | thop=0.1.1 | mamba_ssm=None
5. System RAM        : 50.08 GB free / 52.96 GB total (5.4% used)
6. Disk Space        : 223.52 GB free / 235.68 GB total (/content/Endsem)
------------------------------------------------------------------------------
7. Dataset & Weights Accessibility:
   MVTec LOCO AD     : OK (/content/data)
     train=1778 | val=305 | test_good=575 | test_logical=561 | test_structural=432 (EXCLUDED)
   ImageNet Penalty  : OK (10,000 images at /content/data/imagenet_penalty)
   PDN-S Teacher     : OK (/content/drive/MyDrive/DL_Endsem_Team11/weights/teacher_pdn_s.pt)


In [6]:
#@title 3. Run Phase D (E16_FPN & E17_BiFPN) — Sequential (Clean Solo Latency & Best on L4)
!python scripts/run_experiment.py --config configs/E16_fpn.yaml --mode train --drive-sync-dir "$DRIVE_RUNS"
!python scripts/run_experiment.py --config configs/E17_bifpn.yaml --mode train --drive-sync-dir "$DRIVE_RUNS"

RUN: E16_FPN | mode=train | device=cuda | seed=42
Dataset audit OK: train=1778 | val=305 | test_good=575 | test_logical=561 | structural_excluded=432
Loading pooled train/good and validation/good (img_size=256, cache_in_ram=True)...
  Penalty corpus ready: 10,000 images (batch_size=8)
  Computing streaming teacher channel normalization stats on pooled train/good...
  Starting training: epochs=200, batch_size=8, amp=torch.bfloat16, trainable_params=7,052,929, frozen_params=2,694,144
  Epoch   1/200 | train=10.2719 | val=7.3009 | lr=3.0e-04 | 71.5s <-- BEST
  Epoch   2/200 | train=6.0532 | val=5.1677 | lr=3.0e-04 | 55.9s <-- BEST
  Epoch   3/200 | train=4.7166 | val=4.2130 | lr=3.0e-04 | 55.9s <-- BEST
  Epoch   4/200 | train=3.9959 | val=3.6685 | lr=3.0e-04 | 55.8s <-- BEST
  Epoch   5/200 | train=3.5698 | val=3.4570 | lr=3.0e-04 | 55.8s <-- BEST
  Epoch   6/200 | train=3.3572 | val=3.0837 | lr=3.0e-04 | 55.9s <-- BEST
  Epoch   7/200 | train=3.0817 | val=2.9569 | lr=3.0e-04 | 56.0s <--

In [10]:
#@title 4. Aggregate Phase D vs. Baseline & Phase C Winner Summary Table
!python scripts/aggregate_results.py --dirs experiments "$DRIVE_RUNS" --out-dir "$DRIVE_RUNS/phaseD_summary"

| Run | BB | JB | PP | SB | SC | **Comb Mean** | **Pooled** | Δ Mean | Δ Pooled | Params (M) | GFLOPs | Lat (ms) | Peak VRAM (MiB) | GPU |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :--- |
| E01_PUAD_verify_a2 | 90.99% | 97.58% | 86.73% | 67.87% | 94.64% | **87.56%** | **83.30%** | +0.00 pp | +0.00 pp | 8.058 | 75.9066 | 20.48 | 171.78 | NVIDIA GeForce RTX 4060 Laptop GPU |
| E16_FPN | 91.91% | 98.23% | 98.77% | 68.47% | 94.25% | **90.33%** | **87.84%** | +2.77 pp | +4.54 pp | 9.747 | 88.2741 | 11.585 | 1951.74 | NVIDIA L4 |
| E17_BiFPN | 90.95% | 97.96% | 97.04% | 71.29% | 94.29% | **90.31%** | **83.72%** | +2.75 pp | +0.42 pp | 9.165 | 82.4213 | 11.825 | 2690.08 | NVIDIA L4 |
| E01_PUAD | 90.27% | 95.16% | 78.64% | 67.24% | 88.41% | **83.94%** | **80.07%** | -3.62 pp | -3.23 pp | 8.058 | 75.9066 | 20.217 | 1409.61 | Tesla T4 |
| E01_PUAD_200ep | 90.53% | 96.87% | 98.02% | 68.94% | 93.85% | **89.64%** | **84.07%** |